# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR² dataset of clinicopathological and molecular characteristics in second primary colorectal cancer (CRC) survivors using the `mlcroissant` library.

### Dataset Source

The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and contains structured clinical, demographic, oncological, and molecular data.

In [ ]:
# Install mlcroissant if not already present
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and explore the dataset structure with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load and inspect dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, their fields, and find their unique `@id` values.

In [ ]:
# List all record sets with their @id and field ids
print("Available record sets:")
record_sets = list(dataset.recordsets)
for rs in record_sets:
    print(f"- Record set: {rs['@id']}")
    print("  Fields:")
    for field in rs['field']:
        print(f"    - {field['@id']}: {field.get('name')}")

## 3. Data Extraction

Load data from specific record sets using their `@id` into individual pandas DataFrames for analysis.

In [ ]:
# Identify all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.recordsets]
dataframes = {}
for record_set_id in record_set_ids:
    # Load all records from this record set into a DataFrame
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"{record_set_id}: {list(df.columns)}\n")

# Pick the main patient/data record set for preview: find the record set that includes clinical and demographic data
main_record_set_id = None
for rs in dataset.recordsets:
    fname_list = [field.get('name', '') for field in rs['field']]
    if 'Sex' in fname_list and 'Age' in fname_list:
        main_record_set_id = rs['@id']
        break

if main_record_set_id is not None:
    print(f"Main data record set: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    print("Could not identify the main clinical record set.")

## 4. Exploratory Data Analysis (EDA)

Let's perform some basic processing:
- Filter by age
- Normalize age
- Group by sex
All entity references use their `@id`.

In [ ]:
# Choose numeric field and group field by @id
# Find the field @ids for Age and Sex
age_field_id = None
sex_field_id = None
for rs in dataset.recordsets:
    for field in rs['field']:
        fname = field.get('name', '').lower()
        if fname == 'age':
            age_field_id = field['@id']
        if fname == 'sex':
            sex_field_id = field['@id']
if main_record_set_id is None:
    raise RuntimeError('No main record set identified.')
df = dataframes[main_record_set_id]

# If the column names differ from the field @id, adjust accordingly
if age_field_id not in df.columns:
    print(f"Warning: {age_field_id} not in columns. Using first numeric column as Age.")
    # try to guess
    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) > 0:
        age_field_id = numeric_cols[0]

if sex_field_id not in df.columns:
    print(f"Warning: {sex_field_id} not in columns. Trying to infer 'Sex' column.")
    for c in df.columns:
        if c.lower() == 'sex':
            sex_field_id = c
            break

# Basic filter, normalize, and group
age_threshold = 50
if age_field_id in df.columns:
    filtered_df = df[df[age_field_id] > age_threshold].copy()
    print(f"Filtered records where {age_field_id} > {age_threshold}:")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{age_field_id}_normalized"] = (filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std()
    print(f"Normalized {age_field_id}:")
    display(filtered_df[[age_field_id, f"{age_field_id}_normalized"]].head())
    # Grouping
    if sex_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(sex_field_id)[age_field_id].mean().reset_index()
        print(f"Mean {age_field_id} grouped by {sex_field_id}:")
        display(grouped_df)
else:
    print("No numeric age field found for processing.")

## 5. Visualization

Visualize the age distribution by sex (using the field `@id` as column label).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if age_field_id in df.columns and sex_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(data=df, x=age_field_id, hue=sex_field_id, bins=10, kde=True, multiple='dodge')
    plt.title(f"Age Distribution by {sex_field_id}")
    plt.xlabel(age_field_id)
    plt.ylabel('Count')
    plt.legend(title=sex_field_id)
    plt.show()
else:
    print("Required columns for visualization not found.")

## 6. Conclusion

- This notebook demonstrated loading, exploring, and performing basic data analysis with the FAIR² dataset in accordance with its Croissant schema.
- All references to record sets, fields, and columns were made via their unique `@id`, ensuring reference consistency.
- EDA included age filtering, normalization, and grouping, as well as visualization by sex.
- Further analysis could include examining molecular markers or clinical outcomes using the same approach.